In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

In [ ]:
%pip install transformers==4.57.1 datasets accelerate bitsandbytes peft trl evaluate sacrebleu rouge_score ipywidgets

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
dataset = load_dataset("opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

def rename_columns(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["es"],
    }

dataset = dataset.map(rename_columns, remove_columns=dataset["train"].column_names)

print(dataset)
print(dataset["train"][0])

In [ ]:
def build_prompt(src, tgt=None):
    user = f"Translate this from English to Spanish:\n{src}"
    if tgt is None:
        return f"User: {user}\nAssistant:"
    else:
        return f"User: {user}\nAssistant: {tgt}"

print(build_prompt("Hello"))

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
sample_text = dataset["test"][0]["src"]
reference_text = dataset["test"][0]["tgt"]

baseline_prediction = generate_translation(baseline_model, baseline_tokenizer, sample_text)

print("SOURCE:")
print(sample_text)
print("\nREFERENCE:")
print(reference_text)
print("\nBASELINE OUTPUT:")
print(baseline_prediction)

In [ ]:
test_subset = dataset["test"].select(range(20))

baseline_predictions = []
references = []
sources = []

for example in test_subset:
    src = example["src"]
    tgt = example["tgt"]

    pred = generate_translation(baseline_model, baseline_tokenizer, src)

    sources.append(src)
    references.append(tgt)
    baseline_predictions.append(pred)

print("Done generating baseline predictions.")
print("Example prediction:", baseline_predictions[0])

In [ ]:
bleu = evaluate.load("sacrebleu")

baseline_bleu = bleu.compute(
    predictions=baseline_predictions,
    references=[[ref] for ref in references]
)

print("Baseline BLEU:", baseline_bleu)

In [ ]:
del baseline_model
torch.cuda.empty_cache()

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
max_length = 256

def preprocess(example):
    prompt = (
        "You are a translation system. "
        "Translate the following text from English to Spanish. "
        "Return only the Spanish translation.\n\n"
        f"English: {example['src']}\nSpanish: {example['tgt']}"
    )

    tokens = tokenizer(
        prompt,
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(
    preprocess,
    batched=False,
    remove_columns=dataset["train"].column_names,
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-qlora-translation",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)

trainer.train()

In [ ]:
qlora_predictions = []

for src in sources:
    pred = generate_translation(model, tokenizer, src)
    qlora_predictions.append(pred)

print("QLORA EXAMPLE:")
print(qlora_predictions[0])

In [ ]:
qlora_bleu = bleu.compute(
    predictions=qlora_predictions,
    references=[[ref] for ref in references]
)

print("QLoRA BLEU:", qlora_bleu)

In [ ]:
for i in range(5):
    print("=" * 80)
    print("SOURCE:   ", sources[i])
    print("REFERENCE:", references[i])
    print("BASELINE: ", baseline_predictions[i])
    print("QLORA:    ", qlora_predictions[i])

In [ ]:
model.save_pretrained("./tinyllama_qlora_adapter")
tokenizer.save_pretrained("./tinyllama_qlora_adapter")

In [ ]:
import json

results = {
    "sources": sources,
    "references": references,
    "baseline_predictions": baseline_predictions,
    "qlora_predictions": qlora_predictions,
    "baseline_bleu": baseline_bleu,
    "qlora_bleu": qlora_bleu,
}

with open("translation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)